# SEMEFO-DF: Análisis

Tercer análisis de datos del SEMEFO 1974.
Feliz día de la mujer!


In [18]:
from warnings import warn
import pandas as pd
import numpy as np
from numpy import nan
from statistics import mode
from scipy.stats import zscore
from unidecode import unidecode

import seaborn as sns
import matplotlib.pyplot as plt
from helper_pandas import profile_minimal, profile_by_categorical_column, multicolumn_value_counts_with_filter, top_values_of_series_to_string
from helper_pandas import join_strings_with_nans, flatten_list


def simplify_string(text, text_format='upper'):
    if type(text)==str:
        if text_format=='upper': text = text.upper()
        if text_format=='lower': text = text.lower()
        if text_format=='title': text = text.title()
        if text_format=='capitalize': text = text.capitalize()
        text = unidecode(text)
        text = ' '.join([t for t in text.split()])
    return text


In [2]:
file_review = "data_clean/SEMEFO_BitacoraOnomastica_1974_review.csv"
df = pd.read_csv(file_review)
print('Tamaño original', df.shape)

df['FECHA'] = pd.to_datetime(df['FECHA'])
df = df.drop( df[df['FECHA'].dt.year.isin([1973, 1975])].index )

df.loc[ df['NOMBRE COMPLETO'].isin(['DESCONOCIDO', 'R NACIDO']), 'TIPOS DE RESTOS'] = 'Cadáver (desconocido)'
df['TIPOS DE RESTOS'] = df['TIPOS DE RESTOS'].replace({'Cadáver':'Cadáver (conocido)'})
df.loc[df['NOMBRE COMPLETO']=='R NACIDO', 'TIPOS DE RESTOS'] = 'R nacido'

df['AVERIGUACION PREVIA CON PROCEDENCIA'] = df[['PROCEDENCIA MP Y DEL','AVERIGUACION PREVIA']].apply(join_strings_with_nans, sep='_', axis=1)

df.loc[df['TIPOS DE RESTOS']=='Feto','EDAD'] = nan
df['RANGO EDAD'] = df['RANGO EDAD'].replace({'(5, 10]':'(05, 10]', '(75, 80]':'(75, 105]', '(80, 85]':'(75, 105]', 
                                             '(85, 90]':'(75, 105]', '(90, 95]':'(75, 105]', '(95, 104]':'(75, 105]'})



from pandas.api.types import CategoricalDtype

orden_dias = ['Lunes','Martes','Miércoles','Jueves','Viernes', 'Sábado', 'Domingo',  'S-d' ]
orden_dias = CategoricalDtype(categories=orden_dias, ordered=True)
df['DIA SEMANA'] = df['DIA SEMANA'].fillna('S-d').astype( orden_dias )



df = df[['NOMBRE COMPLETO', 'APELLIDO PATERNO', 'APELLIDO MATERNO', 'NOMBRES PROPIOS',
         'PROCEDENCIA MP Y DEL', 'AVERIGUACION PREVIA CON PROCEDENCIA', 'EXP SEMEFO', 
         'FECHA', 'SEMANA', 'DIA SEMANA', 'TIPOS DE RESTOS', 'EDAD', 'RANGO EDAD', 'SEXO', 'DIAGNOSTICO']]

df.dtypes

Tamaño original (5733, 20)


NOMBRE COMPLETO                                object
APELLIDO PATERNO                               object
APELLIDO MATERNO                               object
NOMBRES PROPIOS                                object
PROCEDENCIA MP Y DEL                           object
AVERIGUACION PREVIA CON PROCEDENCIA            object
EXP SEMEFO                                     object
FECHA                                  datetime64[ns]
SEMANA                                        float64
DIA SEMANA                                   category
TIPOS DE RESTOS                                object
EDAD                                          float64
RANGO EDAD                                     object
SEXO                                           object
DIAGNOSTICO                                    object
dtype: object

In [21]:
cat_diag = "data_raw/Catalogo_Diagnostico_simplificado.csv"
cat_diag = pd.read_csv(cat_diag)
cat_diag = {k:v for k,v in zip(cat_diag['Data'],cat_diag['Equivalencia'])}
cat_diag


diag = df.loc[ (df['SEXO']=='Femenino') | (df['TIPOS DE RESTOS']=='Feto') | (df['EDAD']==0), 'DIAGNOSTICO' ].value_counts().index.to_list()
diag = [d.split('+') for d in diag]
diag = flatten_list(diag)
len(diag)

466

In [27]:
map_diag = dict()
for d in diag:
    d_ = simplify_string(d)
    if d_ in cat_diag.keys():
        #map_diag[d] = cat_diag[d_]  
        pass
    else: 
        map_diag[d] = d_
len(map_diag)

167

In [46]:

df_femi = [df [ df['DIAGNOSTICO'].fillna('').str.lower().str.contains(d) ] for d in ['abort', 'legrado', 'post', 'vag', 'viola', 'post']]
df_femi = pd.concat(df_femi)
df_femi = df_femi[df_femi['SEXO']=='Femenino'].drop_duplicates()
df_femi

,NOMBRE COMPLETO,APELLIDO PATERNO,APELLIDO MATERNO,NOMBRES PROPIOS,PROCEDENCIA MP Y DEL,AVERIGUACION PREVIA CON PROCEDENCIA,EXP SEMEFO,FECHA,SEMANA,DIA SEMANA,TIPOS DE RESTOS,EDAD,RANGO EDAD,SEXO,DIAGNOSTICO
132,ALMANZA JACOBO REFUGIO,ALMANZA,JACOBO,REFUGIO,6A,6A_619,2534,1974-06-24,26.0,Lunes,Cadáver (conocido),27.0,"(25, 30]",Femenino,BN+Aborto
157,ARCOS SOLIS GINA,ARCOS,SOLIS,GINA,5A,5A_743,3188,1974-07-04,27.0,Jueves,Cadáver (conocido),42.0,"(40, 45]",Femenino,S + Aborto Sep
634,CRUZ FLORES SOCORRO,CRUZ,FLORES,SOCORRO,2A,2A_537,1719,1974-04-09,15.0,Martes,Cadáver (conocido),50.0,"(45, 50]",Femenino,PN+Aborto
1281,FIERRO SALAS REBECA,FIERRO,SALAS,REBECA,5A,5A_295,1318,1974-03-17,11.0,Domingo,Cadáver (conocido),22.0,"(20, 25]",Femenino,BNXMANIOBRAS ABORTIVAS
1492,GARCIA FLORES JUANA,GARCIA,FLORES,JUANA,24A,24A_287,919,1974-02-22,8.0,Viernes,Cadáver (conocido),26.0,"(25, 30]",Femenino,Aborto Séptico+S
1583,GONZALEZ CAMACHO GUILLERMINA,GONZALEZ,CAMACHO,GUILLERMINA,10A,10A_482,2002,1974-04-25,17.0,Jueves,Cadáver (conocido),30.0,"(25, 30]",Femenino,BN+ABORTO
2211,ISLAS RIOS SARA,ISLAS,RIOS,SARA,9A,9A_653,1818,1974-02-14,7.0,Jueves,Cadáver (conocido),18.0,"(15, 20]",Femenino,HEM+AN MANIOBRAS ABORT
2298,JAEN ALVAREZ CONCEPCION,JAEN,ALVAREZ,CONCEPCION,5A,5A_1602,4015,1974-08-25,34.0,Domingo,Cadáver (conocido),26.0,"(25, 30]",Femenino,BN+ABORTO
2895,MARTINEZ HERNANDEZ JOSEFINA,MARTINEZ,HERNANDEZ,JOSEFINA,5A,5A_733,3165,1974-07-02,27.0,Martes,Cadáver (conocido),22.0,"(20, 25]",Femenino,Cuadro Séptico por aborto
3479,PEREA HERRERA DELFINA,PEREA,HERRERA,DELFINA,29A,29A_92,3076,1974-06-25,26.0,Martes,Cadáver (conocido),30.0,"(25, 30]",Femenino,AN+ ABORTO


In [52]:
df_femi[['EDAD', 'FECHA', 'DIAGNOSTICO','PROCEDENCIA MP Y DEL', ]].sort_values(['EDAD','DIAGNOSTICO'])

,EDAD,FECHA,DIAGNOSTICO,PROCEDENCIA MP Y DEL
5705,0.0,1974-03-21,AXASPPOST PART,10A
4841,6.0,1974-07-12,ASB+Violación,18A
2211,18.0,1974-02-14,HEM+AN MANIOBRAS ABORT,9A
1281,22.0,1974-03-17,BNXMANIOBRAS ABORTIVAS,5A
2895,22.0,1974-07-02,Cuadro Séptico por aborto,5A
3735,24.0,1974-03-05,Aborto incomp+AN,22A
1488,25.0,1974-02-24,BN+Post Legrado,5A
1492,26.0,1974-02-22,Aborto Séptico+S,24A
2298,26.0,1974-08-25,BN+ABORTO,5A
132,27.0,1974-06-24,BN+Aborto,6A
